<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Access Land Surface Temperature Data From ECOSTRESS**

 <span style='color:#ff6666'><font size="5"> **About the data**

The ECOsystem Spaceborne Thermal Radiometer Experiment on Space Station (ECOSTRESS) mission measures the temperature of plants to better understand how much water plants need and how they respond to stress. ECOSTRESS is attached to the International Space Station (ISS) and collects data globally between 52° N and 52° S latitudes. **Source:** [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/lpcloud-eco-l2-lste-002).


 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)

 <span style='color:#ff6666'><font size="5"> **Objectives**

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3">Stream subset of data

 <span style='color:#ff6666'><font size="5"> **References**

<font size="3"> **Hook, S., &amp; Hulley, G.(2022)**. <i>ECOSTRESS Swath Land Surface Temperature and Emissivity Instantaneous L2 Global 70 m v002</i> [Data set]. NASA Land Processes Distributed Active Archive Center. https://doi.org/10.5067/ECOSTRESS/ECO_L2_LSTE.002


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import pydap
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
import numpy as np
from pydap.client import to_netcdf as dap_to_netcdf

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via earthaccess as demonstrated below. You must have a valid EDL account. There are two strategies for authenticating with `earthaccess`:

1. `strategy="interactive"`. This will promt your edl username-password.
2. `strategy="netrc"`. Use this if the notebook is running on an environment where a `.netrc` with your credentials is recoverable. T

<font size="3.5"> Below the default will be `netrc`, assuming the user has executed the notebook **Authenticate.ipynb**. If not, you can change the strategy to `"interactive"`.



In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

<font size="3"> We query NASA's CMR to identify remote files that intersect the following geographical area (bounding box) covering the following time range

- <font size="3"> -128.85 < longitude < -107.05, and 41.1 < latitude < 46.68
- <font size="3"> 03/01 - 04/30 (2025)




In [ ]:
ECOSTRESS_ccid = "C2076114664-LPCLOUD"
bounding_box = [-128.847656, 41.112469, -107.050781, 46.679594]
time_range = [dt.datetime(2025, 3, 1), dt.datetime(2025, 4, 30)]

cmr_urls = get_cmr_urls(ccid=ECOSTRESS_ccid, bounding_box = bounding_box, limit=1000, time_range=time_range) # limit by default = 50

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Accessing Metadata-ONLY with PyDAP**

<font size="3"> We can access <span style='color:#ff6666'>**OPeNDAP**</span>-produced metadata to identify the variables of interest. In particular those associated with latitude and longitude values

<font size="3"> Below need to request the <span style='color:#ff6666'>**DAP4**</span> metadata from the remote server. To specify the protocol, we have 2 options:

**1.** <font size="3"> Replace "https" with "dap4" in the URL. This works when using `Xarray`:
```python
open_url(url.replace("https","dap4"), ...)
```
**2**. <font size="3"> Specify the protocol directly (**does not work with Xarray**)
```python
open_url(url, protocol='dap4', ...)
```

<font size="3"> Below we follow option **2)** above.


In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

### Subset data

First, we would like to subset data based on the criteria:

* Daylight data
* QAPercentCloudCover <30% 
* and QAPercentGoodQuality> 70%


For that, we need to download only 3 variables:

```python
    ["/L2 LSTE Metadata/QAPercentCloudCover", 
     "/L2 LSTE Metadata/QAPercentGoodQuality",
     "/StandardMetadata/DayNightFlag"]
```

In [ ]:
output_path = 'data/'

## Stream data into local directory

Stream each remote file into an individual file, since these cannot be aggregated.


In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, output_path = output_path, 
              keep_variables=["/L2 LSTE Metadata/QAPercentCloudCover", 
                              "/L2 LSTE Metadata/QAPercentGoodQuality",
                              "/StandardMetadata/DayNightFlag"]
             )

## Inspect all downloaded files

<font size="3"> Data has been already downloaded, and we can further filter to identify remote granules by

* <font size="3"> Daylight data
* <font size="3"> QAPercentCloudCover <30% 
* <font size="3"> and QAPercentGoodQuality> 70%

<font size="3"> Then we will update our list of URLs to download from, based on the remote files that
satisfy the above criteria

<font size="3"> **NOTE**:  The files cannot be aggregated!!


In [ ]:
final_urls = []
for i in range(len(cmr_urls)):
    local_file = output_path+cmr_urls[i].split("/")[-1]+".nc4"
    dst = xr.open_datatree(local_file).load()
    if dst['L2 LSTE Metadata/QAPercentCloudCover'].values < 30 and dst["L2 LSTE Metadata/QAPercentGoodQuality"] > 70 and dst["/StandardMetadata/DayNightFlag"] == 'Day':
         final_urls.append(cmr_urls[i])

print("Total remote granules to download: ", len(final_urls))

### Declare final variables to download

<font size="3"> At this point, it is crucial to remove any ECOSTRESS data downloaded into the data_output directory. For that open a terminal and navigate to the data directory


```shell
cd data
rm ECOv002_L2*.nc4
```

In [ ]:
import os
import glob

fnames = [output_path+f"{fname.split('/')[-1]}.nc4" for fname in cmr_urls]

for filename in fnames:
    try:
        os.remove(filename)
    except FileNotFoundError:
        print(f"The file '{filename}' is not in there anymore")    

In [ ]:
# define all variables of interest that our final download will have

keep_vars = ["/StandardMetadata/EastBoundingCoordinate", 
             "/StandardMetadata/SouthBoundingCoordinate", 
             "/StandardMetadata/NorthBoundingCoordinate",
             "/StandardMetadata/WestBoundingCoordinate",
             "/StandardMetadata/DayNightFlag",
             "/StandardMetadata/ImagePixels",
             "/StandardMetadata/ImagePixelSpacing",
             "/StandardMetadata/ImageLines",
             "/StandardMetadata/RangeBeginningDate",
             "/StandardMetadata/RangeBeginningTime",
             "/StandardMetadata/RangeEndingDate",
             "/L2 LSTE Metadata/QAPercentCloudCover",
             "/L2 LSTE Metadata/QAPercentGoodQuality",
             "/SDS/QC", "/SDS/LST",
            ]

### Data Download

<font size="3"> Do not forget to now use the final_urls list, which points to our granules of interest.


In [ ]:
%%time
dap_to_netcdf(final_urls, session=my_session, output_path = output_path,
              keep_variables=keep_vars,
             )

### Inspect local files

<font size="3"> Data has been already downloaded! 

<font size="3"> **NOTE:** These datasets cannot be aggregated


In [ ]:
local_file = output_path+final_urls[0].split("/")[-1]+".nc4"
dst = xr.open_datatree(local_file)
dst
